# 🧪 Sesión 26 · Transacciones ACID, Aislamiento y Rollback

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué pasa si falla la segunda mitad de una operación?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/libros/1                 -> 200 {"ejemplares":1}
POST /api/prestamos {"libro_id":1,"usuario":"ana"}
                                   -> 201 {"prestamo_id":1,"quedan":0}
GET /api/libros/1                 -> 200 {"ejemplares":0}
POST /api/prestamos {"libro_id":1,"usuario":"luis"}
                                   -> 409 {"error":"sin ejemplares"}
GET /api/libros/1                 -> 200 {"ejemplares":0}  # no bajó a -1
```

_Si el segundo préstamo descontara antes de validar, el stock quedaría en -1. La transacción lo impide._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Agrupar varias escrituras en una transacción.
- Hacer rollback cuando una condición de negocio falla.
- Verificar que el stock no queda inconsistente.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El libro inicia con un ejemplar → HTTP 200
- [ ] El primer préstamo se completa → HTTP 201
- [ ] El stock quedó en cero → HTTP 200
- [ ] El segundo préstamo falla con 409 → HTTP 409
- [ ] El stock no quedó negativo → HTTP 200
- [ ] La operación usa una transacción

## 🧩 El reto

Implementa un préstamo de libros que descuente stock y registre el préstamo en una sola transacción.

**Requisitos del contrato**

- La base inicia con el libro 1, «Rayuela», con un ejemplar disponible.
- POST /api/prestamos descuenta un ejemplar y registra el préstamo.
- Si no hay ejemplares, responde 409 y no modifica el stock.
- GET /api/libros/{id} muestra los ejemplares restantes.
- No puede quedar stock negativo ni un préstamo sin descuento.

> **Pista conceptual:** Valida y escribe dentro de la misma transacción. Si haces `commit` antes de comprobar la condición, ya no puedes deshacer el descuento.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion26.md` con el caso de rollback.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [10]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [11]:
%%writefile app.py

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import sqlite3

app = FastAPI(
    title="Biblioteca API",
    version="1.0.0"
)

DATABASE = "biblioteca.db"


# Conexión a la base de datos
def conectar():
    conexion = sqlite3.connect(DATABASE, timeout=10)
    conexion.row_factory = sqlite3.Row
    return conexion


# Crear las tablas e insertar el libro inicial
def inicializar_bd():
    conexion = conectar()

    try:
        conexion.execute("""
            CREATE TABLE IF NOT EXISTS libros (
                id INTEGER PRIMARY KEY,
                titulo TEXT NOT NULL,
                ejemplares INTEGER NOT NULL CHECK (ejemplares >= 0)
            )
        """)

        conexion.execute("""
            CREATE TABLE IF NOT EXISTS prestamos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                libro_id INTEGER NOT NULL,
                usuario TEXT NOT NULL,
                FOREIGN KEY (libro_id) REFERENCES libros(id)
            )
        """)

        conexion.execute("""
            INSERT OR IGNORE INTO libros (id, titulo, ejemplares)
            VALUES (1, 'Rayuela', 1)
        """)

        conexion.commit()

    finally:
        conexion.close()


# Preparar la base de datos
inicializar_bd()


# Modelo para recibir los datos del préstamo
class PrestamoEntrada(BaseModel):
    libro_id: int
    usuario: str


# Consultar los ejemplares disponibles
@app.get("/api/libros/{id}")
def obtener_libro(id: int):
    conexion = conectar()

    try:
        libro = conexion.execute(
            "SELECT id, titulo, ejemplares FROM libros WHERE id = ?",
            (id,)
        ).fetchone()

        if libro is None:
            return JSONResponse(
                status_code=404,
                content={"error": "libro no encontrado"}
            )

        return {
            "id": libro["id"],
            "titulo": libro["titulo"],
            "ejemplares": libro["ejemplares"]
        }

    finally:
        conexion.close()


# Registrar un préstamo mediante una transacción
@app.post("/api/prestamos", status_code=201)
def crear_prestamo(datos: PrestamoEntrada):

    conexion = conectar()

    try:
        # Iniciar la transacción
        conexion.execute("BEGIN IMMEDIATE")

        # Consultar el libro dentro de la transacción
        libro = conexion.execute(
            "SELECT ejemplares FROM libros WHERE id = ?",
            (datos.libro_id,)
        ).fetchone()

        if libro is None:
            conexion.rollback()

            return JSONResponse(
                status_code=404,
                content={"error": "libro no encontrado"}
            )

        # Validar que existan ejemplares
        if libro["ejemplares"] <= 0:
            conexion.rollback()

            return JSONResponse(
                status_code=409,
                content={"error": "sin ejemplares"}
            )

        # Descontar un ejemplar
        conexion.execute(
            """
            UPDATE libros
            SET ejemplares = ejemplares - 1
            WHERE id = ? AND ejemplares > 0
            """,
            (datos.libro_id,)
        )

        # Registrar el préstamo
        cursor = conexion.execute(
            """
            INSERT INTO prestamos (libro_id, usuario)
            VALUES (?, ?)
            """,
            (datos.libro_id, datos.usuario)
        )

        prestamo_id = cursor.lastrowid

        # Confirmar ambas operaciones
        conexion.commit()

        # Consultar el stock actualizado
        libro_actualizado = conexion.execute(
            "SELECT ejemplares FROM libros WHERE id = ?",
            (datos.libro_id,)
        ).fetchone()

        return {
            "prestamo_id": prestamo_id,
            "libro_id": datos.libro_id,
            "quedan": libro_actualizado["ejemplares"]
        }

    except Exception:
        # Deshacer los cambios si ocurre un error
        conexion.rollback()
        raise

    finally:
        conexion.close()


Writing app.py


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [12]:
from verificar import probar_http

casos = [{'descripcion': 'El libro inicia con un ejemplar',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'id': 1, 'ejemplares': 1}},
 {'descripcion': 'El primer préstamo se completa',
  'metodo': 'POST',
  'ruta': '/api/prestamos',
  'json': {'libro_id': 1, 'usuario': 'ana'},
  'estado': 201,
  'json_contiene': {'libro_id': 1, 'quedan': 0}},
 {'descripcion': 'El stock quedó en cero',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'ejemplares': 0}},
 {'descripcion': 'El segundo préstamo falla con 409',
  'metodo': 'POST',
  'ruta': '/api/prestamos',
  'json': {'libro_id': 1, 'usuario': 'luis'},
  'estado': 409},
 {'descripcion': 'El stock no quedó negativo',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'ejemplares': 0}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 5 casos · app.py
✅ 01. El libro inicia con un ejemplar
✅ 02. El primer préstamo se completa
✅ 03. El stock quedó en cero
✅ 04. El segundo préstamo falla con 409
✅ 05. El stock no quedó negativo
------------------------------------------------------------------------------
RESULTADO: 5/5 casos correctos. ¡Resultado logrado!


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | 	Utilizar Python 3.13, FastAPI y SQLite en Google Colab. Se trabajará con una base de datos para administrar libros y préstamos, cuidando que los ejemplares disponibles no queden en cantidades negativas. |
| **Rol** | Actuar como un desarrollador de aplicaciones web especializado en Python, FastAPI y bases de datos, encargado de implementar transacciones seguras y controlar los errores durante las operaciones.|
| **Especificación** | Crear las rutas GET /api/libros/{id} y POST /api/prestamos. El sistema debe consultar los ejemplares disponibles, descontar uno cuando se realice un préstamo y registrar al usuario. Si no hay ejemplares, debe responder con HTTP 409 y utilizar rollback para evitar modificar el stock. También debe evitar que se registre un préstamo sin descontar un ejemplar.|
| **Output** | Entregar el archivo app.py completo y funcional, utilizando SQLite y transacciones con BEGIN, COMMIT y ROLLBACK. La implementación debe cumplir con las cinco pruebas HTTP del verificador y mantener los datos consistentes.|

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


In [13]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["commit|rollback"], descripcion="La operación usa una transacción"))
resumen(resultados)


✅ La operación usa una transacción
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion26.md` a `auditorias/auditoria_sesion26.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Me costó más conseguir que el segundo préstamo fuera rechazado sin modificar la cantidad de ejemplares. Tuve que asegurarme de validar el stock antes de registrar el préstamo y utilizar una transacción para evitar que los datos quedaran inconsistentes.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí utilizar SQLite para guardar los libros y los préstamos en tablas separadas. También utilicé BEGIN IMMEDIATE para controlar las operaciones de escritura y evitar que dos solicitudes descuenten el mismo ejemplar disponible.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   Comprendí que no basta con descontar un ejemplar y registrar el préstamo por separado, porque si ocurre un error durante la segunda operación los datos pueden quedar inconsistentes. Por eso utilicé ROLLBACK para cancelar los cambios cuando ocurre un problema.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
